# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Левченко Денис'
ГРУПА        = 'ШІ-33'
НОМЕР        = 4

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 4   ·   Левченко Денис   ·   ШІ-33

Ваші вісім стовпців:
   1. Телефон
   2. Ускладнення
   3. Перебіг
   4. Виписка
   5. Ліжко днів
   6. Д-дим1
   7. Фібрин2
   8. МНВ2

Ваша пара для аналізу зв’язку: ступінь КТ та ЛН

Ваше додаткове питання:
   У чому різниця між шкалою інтервалів і шкалою відношень? Знайдіть у файлі по одному прикладу кожної та поясніть, яка операція допустима для другої, але не для першої.


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Ускладнення,Перебіг,Виписка,Ліжко днів,Д-дим1,Фібрин2,МНВ2
0,0958469788 дружина Ганна,"тромбоз підколінної артерії, гангрена? ТЕЛА? СЛН ГРДС",важкий,2022-03-10 00:00:00,14,NaN,2.30,239
1,"0671778486, 0994887663 дружина Оксана",NaN,важкий,2022-03-16 00:00:00,22,3316,3.40,1.08
2,"0958768045, чоловік Степан 0951784726",NaN,важкий,2022-01-18 00:00:00,20,NaN,3.27,1.18
3,0959819331. дочка Валентина 0960845205,NaN,важкий,2022-01-13 00:00:00,20,NaN,4.00,1.07
4,"0952294866, дочка Дарина 0678360161",NaN,важкий,2022-03-11 00:00:00,27,NaN,3.37,2.18
5,0959922983 сестра Софія,"Респірат дестрес синдром, емфізема легень, СЛН",важкий,2021-12-26 00:00:00,9(5 у реан),8000,3.00,2015-01-01 00:00:00
6,"0998073702, дружина Наталія 0998382226",NaN,важкий,2022-05-11 00:00:00,18,183,3.12,1.09
7,"0637258771, син Сергій 0968413301",NaN,серед важ,2022-05-28 00:00:00,10,3028,2.40,1.09


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Телефон      120 non-null    object 
 1   Ускладнення  61 non-null     object 
 2   Перебіг      121 non-null    object 
 3   Виписка      120 non-null    object 
 4   Ліжко днів   120 non-null    object 
 5   Д-дим1       59 non-null     object 
 6   Фібрин2      101 non-null    float64
 7   МНВ2         76 non-null     object 
dtypes: float64(1), object(7)
memory usage: 7.7+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Ускладнення
    непорожніх 61 з 121 | унікальних 35 | типи: str×61
    приклади: ["'тромбоз підколінної артерії, гангрена? ТЕЛА? СЛН ГРДС'", "'Респірат дестрес синдром, емфізема легень, СЛН'", "'Респірат дестрес синдром, емфізема легень, СЛН ТІШ'", "'Гострий респіраторний дистрес синдром,ТЕЛА?Прогресуюча серц-леген недос'"]

--- Перебіг
    непорожніх 121 з 121 | унікальних 3 | типи: str×121
    приклади: ["'важкий'", "'серед важ'", "'вкрай важкий '"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- Ліжко днів
    непорожніх

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'Телефон': ('ідентифікатор', 'номер телефону родича разом з іменем: унікальний для кожного '
                'пацієнта і не є величиною, його не можна порівнювати чи усереднювати'),
    'Ускладнення': ('номінальна', 'вільний текст із назвами ускладнень (ГРДС, ТЕЛА, набряк легень): '
                    'значення можна лише порівняти на рівність, порядку між ними немає'),
    'Перебіг': ('порядкова', 'рівні «серед важ» < «важкий» < «вкрай важкий» упорядковані за тяжкістю, '
                'але відстані між сусідніми рівнями невідомі'),
    'Виписка': ('інтервальна', 'дата: різниця двох дат (кількість днів) має сенс, а нуль календаря '
                'умовний, тому «удвічі пізніша дата» не має змісту'),
    'Ліжко днів': ('відношень', 'кількість днів у стаціонарі має природний нуль, а відношення мають '
                   'сенс: 20 днів — удвічі довше за 10'),
    'Д-дим1': ('відношень', 'концентрація D-димеру в нг/мл: нуль означає відсутність речовини, '
               'відношення мають сенс, а одиницю вимірювання обрано умовно'),
    'Фібрин2': ('відношень', 'концентрація фібриногену в г/л має природний нуль, тому «удвічі більше '
                'фібриногену» має фізичний зміст'),
    'МНВ2': ('абсолютна', 'МНВ — безрозмірне відношення протромбінового часу пацієнта до норми: '
             'природні і нуль, і одиниця (1,0 — норма)'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,номер телефону родича разом з іменем: унікальний для кож...
1,Ускладнення,номінальна,"вільний текст із назвами ускладнень (ГРДС, ТЕЛА, набряк ..."
2,Перебіг,порядкова,рівні «серед важ» < «важкий» < «вкрай важкий» упорядкова...
3,Виписка,інтервальна,"дата: різниця двох дат (кількість днів) має сенс, а нуль..."
4,Ліжко днів,відношень,"кількість днів у стаціонарі має природний нуль, а віднош..."
5,Д-дим1,відношень,концентрація D-димеру в нг/мл: нуль означає відсутність ...
6,Фібрин2,відношень,"концентрація фібриногену в г/л має природний нуль, тому ..."
7,МНВ2,абсолютна,МНВ — безрозмірне відношення протромбінового часу пацієн...


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
ПОРЯДОК = {'Перебіг': ['серед важ', 'важкий', 'вкрай важкий']}


def як_число(v):
    if isinstance(v, (int, float)):
        return v
    return pd.to_numeric(str(v).split(';')[0].split('(')[0].strip(), errors='coerce')


def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р
    if ш == 'номінальна':
        частоти = з.astype(str).str.strip().value_counts()
        р['мода'] = частоти.index[0][:40] + '…'
        р['частота моди'] = int(частоти.iloc[0])
        return р
    if ш == 'порядкова':
        рівні = pd.Categorical(з.astype(str).str.strip(), categories=ПОРЯДОК[стовпець], ordered=True)
        коди = pd.Series(рівні.codes)
        р['частоти'] = {к: int(н) for к, н in pd.Series(рівні).value_counts(sort=False).items()}
        р['мода'] = рівні.categories[коди.mode()[0]]
        р['медіана'] = рівні.categories[int(коди.median())]
        р['Q1–Q3'] = (рівні.categories[int(коди.quantile(0.25, interpolation='lower'))],
                      рівні.categories[int(коди.quantile(0.75, interpolation='lower'))])
        return р
    if ш == 'інтервальна':
        дати = pd.to_datetime(з, errors='coerce').dropna()
        р['не дати'] = len(з) - len(дати)
        р['медіана'] = str(дати.median().date())
        р['середнє'] = str(дати.mean().date())
        р['ст. відхилення, днів'] = round(дати.std() / pd.Timedelta(days=1), 1)
        return р
    числа = pd.to_numeric(з.map(як_число), errors='coerce').dropna()
    р['не числа'] = len(з) - len(числа)
    р['мода'] = round(float(числа.mode()[0]), 3)
    р['медіана'] = round(float(числа.median()), 3)
    р['Q1–Q3'] = (round(float(числа.quantile(0.25)), 3), round(float(числа.quantile(0.75)), 3))
    р['середнє'] = round(float(числа.mean()), 3)
    р['ст. відхилення'] = round(float(числа.std()), 3)
    р['коеф. варіації'] = round(float(числа.std() / числа.mean()), 3)
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120}

--- Ускладнення  (номінальна)
    {'непорожніх': 61, 'унікальних': 35, 'мода': 'Гострий респіраторний дистрес синдром. Г…', 'частота моди': 14}

--- Перебіг  (порядкова)
    {'непорожніх': 121, 'унікальних': 3, 'частоти': {'серед важ': 30, 'важкий': 90, 'вкрай важкий': 1}, 'мода': 'важкий', 'медіана': 'важкий', 'Q1–Q3': ('важкий', 'важкий')}

--- Виписка  (інтервальна)
    {'непорожніх': 120, 'унікальних': 90, 'не дати': 2, 'медіана': '2022-03-15', 'середнє': '2022-01-29', 'ст. відхилення, днів': 109.5}

--- Ліжко днів  (відношень)
    {'непорожніх': 120, 'унікальних': 40, 'не числа': 0, 'мода': 10.0, 'медіана': 15.0, 'Q1–Q3': (10.0, 22.0), 'середнє': 17.883, 'ст. відхилення': 12.937, 'коеф. варіації': 0.723}

--- Д-дим1  (відношень)
    {'непорожніх': 59, 'унікальних': 48, 'не числа': 0, 'мода': 10000.0, 'медіана': 1581.0, 'Q1–Q3': (672.99, 5300.0), 'середнє': 3529.674, 'ст. відхилення': 3690.366, 'коеф. варіації

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ускладнення,61,35,Гострий респіраторний дистрес синдром. Гостра серц-леген...,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Перебіг,121,3,важкий,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-09-07 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ліжко днів,120.0,40.0,10.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Д-дим1,59.0,48.0,10000.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Фібрин2,101.0,NaN,NaN,NaN,4.220396,1.884609,0.99,2.7,3.92,5.05,11.1
МНВ2,76.0,51.0,1.24,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
телефони = pd.to_numeric(df['Телефон'], errors='coerce')
перебіг_абетка = df['Перебіг'].astype('category').cat.codes
мнв = pd.to_numeric(df['МНВ2'], errors='coerce')
д_дим = pd.to_numeric(df['Д-дим1'], errors='coerce')

БЕЗГЛУЗДІ = [
    ('Телефон', 'середнє', f'{телефони.mean():.0f}',
     f'середнє телефонних номерів — не номер і не величина; до того ж pandas узяв лише '
     f'{телефони.notna().sum()} номери, записані числом без нуля на початку'),
    ('Перебіг', 'середнє кодів категорій', f'{перебіг_абетка.mean():.3f}',
     'cat.codes нумерує рівні за абеткою («важкий» = 0, «вкрай важкий» = 1, «серед важ» = 2), '
     'а не за тяжкістю, тож середнє залежить від правопису, а не від стану пацієнтів'),
    ('МНВ2', 'середнє', f'{мнв.mean():.2f}',
     f'середнє {мнв.mean():.2f} означало б тяжку коагулопатію, але його тягне вгору одне неможливе '
     f'значення {мнв.max():.0f} (помилка введення); медіана — {мнв.median():.2f}'),
    ('Д-дим1', 'середнє', f'{д_дим.mean():.0f}',
     f'{(д_дим == 10000).sum()} значень дорівнюють рівно 10000 — це верхня межа аналізатора, справжні '
     f'значення більші, а записи на кшталт «4442; 1084» pandas мовчки відкинув'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,середнє,758798950,середнє телефонних номерів — не номер і не величина; до ...
1,Перебіг,середнє кодів категорій,0.504,"cat.codes нумерує рівні за абеткою («важкий» = 0, «вкрай..."
2,МНВ2,середнє,4.41,"середнє 4.41 означало б тяжку коагулопатію, але його тяг..."
3,Д-дим1,середнє,3567,11 значень дорівнюють рівно 10000 — це верхня межа аналі...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: ступінь КТ × ЛН

ступінь КТ: шкала — не у вашому списку
ЛН: шкала — не у вашому списку

  ступінь КТ      ЛН
0    сердній  ІІІ-IV
1   середній  ІІ-ІІІ
2  критичний      ІІ
3  критичний  ІІІ-IV
4        NaN  ІІ-ІІІ
5   середній  ІІ-ІІІ


In [12]:
ОБРАНА_МІРА = 'рангова кореляція Спірмена'
ЧОМУ_САМЕ_ВОНА = ('обидві змінні порядкові: у ступеня КТ і ступеня ЛН є природний порядок, але '
                  'відстані між рівнями невідомі, тому порівнюємо лише ранги, а не самі коди')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

КТ_РІВНІ = {'легкий': 1, 'легка': 1, 'середній': 2, 'сердній': 2, 'серед ступ': 2,
            'серед ступеня': 2, 'важкий': 3, 'вкрай важкий': 4, 'критичний': 5}
ЛН_РІВНІ = {'I': 1, 'I-II': 2, 'II': 3, 'II-III': 4, 'III': 5, 'III-IV': 6}

кт_рівень = df[A].str.strip().map(КТ_РІВНІ)
лн_рівень = df[B].str.strip().str.replace('І', 'I').map(ЛН_РІВНІ)
assert кт_рівень.notna().sum() == df[A].notna().sum(), 'є нерозпізнаний ступінь КТ'
assert лн_рівень.notna().sum() == df[B].notna().sum(), 'є нерозпізнаний ступінь ЛН'

пара = pd.DataFrame({A: кт_рівень, B: лн_рівень}).dropna()
спірмен = пара[A].rank().corr(пара[B].rank())

print('Пацієнтів з обома значеннями:', len(пара))
print(pd.crosstab(пара[A], пара[B]))
print()
print('Кореляція Спірмена:', round(спірмен, 3))

Пацієнтів з обома значеннями: 42
ЛН          1  2  3  4  5  6
ступінь КТ                  
1.0         4  1  1  1  0  0
2.0         3  5  4  5  0  1
3.0         0  2  0  3  4  1
4.0         0  0  0  1  3  0
5.0         0  0  1  0  1  1

Кореляція Спірмена: 0.642


In [13]:
є_обидва = df[A].notna() & df[B].notna()
абетка_кт = df[A].astype('category').cat.codes[є_обидва]
абетка_лн = df[B].astype('category').cat.codes[є_обидва]

print('Пірсон на кодах за абеткою:        ', round(абетка_кт.corr(абетка_лн), 3))
print('Пірсон на кодах рівнів 1..5 і 1..6:', round(пара[A].corr(пара[B]), 3))
print('Пірсон, якщо КТ закодувати 1, 4, 9, 16, 25:', round((пара[A] ** 2).corr(пара[B]), 3))
print('Спірмен для кодів 1, 4, 9, 16, 25: ', round((пара[A] ** 2).rank().corr(пара[B].rank()), 3))
print()
print('Пірсон вимірює лінійний зв’язок і вважає відстані між кодами справжніми. Для порядкових '
      'шкал ці відстані вигадані: за абеткою «важкий» стоїть перед «легкий», а інший, так само '
      'законний вибір кодів змінює коефіцієнт. Спірмен залежить лише від порядку, тому не змінюється.')

Пірсон на кодах за абеткою:         -0.456
Пірсон на кодах рівнів 1..5 і 1..6: 0.6
Пірсон, якщо КТ закодувати 1, 4, 9, 16, 25: 0.547
Спірмен для кодів 1, 4, 9, 16, 25:  0.642

Пірсон вимірює лінійний зв’язок і вважає відстані між кодами справжніми. Для порядкових шкал ці відстані вигадані: за абеткою «важкий» стоїть перед «легкий», а інший, так само законний вибір кодів змінює коефіцієнт. Спірмен залежить лише від порядку, тому не змінюється.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
ПІДГОТОВКА = {
    'Телефон': 'ідентифікатор — у модель не беремо, стовпець видаляємо',
    'Ускладнення': 'номінальна — розбиваємо текст на окремі ускладнення й робимо 0/1-ознаку для '
                   'кожного, без порядку',
    'Перебіг': 'порядкова — цілі коди із заданим вручну порядком: серед важ = 0, важкий = 1, '
               'вкрай важкий = 2',
    'Виписка': 'інтервальна — переходимо до тривалості в днях від поступлення, потім стандартизуємо',
    'Ліжко днів': 'відношень — виймаємо число з тексту, логарифмуємо через довгий правий хвіст '
                  'і стандартизуємо',
    'Д-дим1': 'відношень — виймаємо число, позначаємо цензуровані 10000, логарифмуємо й стандартизуємо',
    'Фібрин2': 'відношень — нормалізація min-max після перевірки, що max не дорівнює min',
    'МНВ2': 'абсолютна — прибираємо неможливі значення (дата Excel, 239) і нормалізуємо min-max',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,"ідентифікатор — у модель не беремо, стовпець видаляємо"
1,Ускладнення,номінальна,номінальна — розбиваємо текст на окремі ускладнення й ро...
2,Перебіг,порядкова,порядкова — цілі коди із заданим вручну порядком: серед ...
3,Виписка,інтервальна,інтервальна — переходимо до тривалості в днях від поступ...
4,Ліжко днів,відношень,"відношень — виймаємо число з тексту, логарифмуємо через ..."
5,Д-дим1,відношень,"відношень — виймаємо число, позначаємо цензуровані 10000..."
6,Фібрин2,відношень,"відношень — нормалізація min-max після перевірки, що max..."
7,МНВ2,абсолютна,"абсолютна — прибираємо неможливі значення (дата Excel, 2..."


In [15]:
перебіг_код = df['Перебіг'].str.strip().map({'серед важ': 0, 'важкий': 1, 'вкрай важкий': 2})

УСКЛАДНЕННЯ = {'ГРДС': 'ГРДС|дистрес', 'ТЕЛА': 'ТЕЛА', 'набряк легень': 'набряк легень', 'шок': 'шок'}
ускладнення_01 = pd.DataFrame({н: df['Ускладнення'].fillna('').str.contains(в, case=False)
                               for н, в in УСКЛАДНЕННЯ.items()}).astype(int)

тривалість = (pd.to_datetime(df['Виписка'], errors='coerce')
              - pd.to_datetime(df['Поступлення'], errors='coerce')).dt.days
тривалість_z = (тривалість - тривалість.mean()) / тривалість.std()


def нормалізувати(x):
    if x.max() == x.min():
        raise ValueError('сталий стовпець: max дорівнює min, знаменник нульовий')
    return (x - x.min()) / (x.max() - x.min())


фібрин_01 = нормалізувати(df['Фібрин2'])

print('Перебіг (порядкова), коди:', перебіг_код.value_counts().sort_index().to_dict())
print('Ускладнення (номінальна), скільки одиниць:', ускладнення_01.sum().to_dict())
print('Виписка (інтервальна) → тривалість: середнє', round(тривалість.mean(), 1), 'дня;',
      f'після стандартизації середнє {abs(тривалість_z.mean()):.3f} і std {тривалість_z.std():.3f}')
print('Фібрин2 (відношень) після нормалізації: min', фібрин_01.min(), '| max', фібрин_01.max())

Перебіг (порядкова), коди: {0: 30, 1: 90, 2: 1}
Ускладнення (номінальна), скільки одиниць: {'ГРДС': 44, 'ТЕЛА': 13, 'набряк легень': 17, 'шок': 28}
Виписка (інтервальна) → тривалість: середнє 18.9 дня; після стандартизації середнє 0.000 і std 1.000
Фібрин2 (відношень) після нормалізації: min 0.0 | max 1.0


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [18]:
ВИПРАВЛЕННЯ = [
    (1, 'середнє від кодів номінальної ознаки «помер/жив» (1 = помер, 2 = вижив)',
     'коди 1 і 2 — лише мітки; 1.50 означає тільки, що померлих і живих майже порівну, а не стан '
     '«посередині між смертю й одужанням»',
     'частки та мода: летальність = (df["помер/жив"] == 1).mean() = 50.4 %'),
    (2, 'частоти номінальної ознаки пораховано на неочищених мітках',
     '«ж» і «жін», «ч» і «чол» — та сама стать, записана по-різному; чотири «статі» — дефект '
     'введення, а не результат',
     'звести мітки до першої літери після strip(): ж — 75, ч — 46'),
    (3, 'порядкову шкалу відсортовано за абеткою, коди взято з алфавітного порядку, варіанти '
        'правопису не об’єднано',
     'порядок «важкий … середній» абетковий, а середнє кодів 5.9 → «легкий» не має змісту: для '
     'порядкової шкали середнє недопустиме',
     'звести правопис до 5 рівнів, задати порядок вручну (легкий < середній < важкий < вкрай важкий '
     '< критичний) і взяти медіану — «середній»'),
    (4, 'pd.to_numeric(errors="coerce") мовчки викинув 16 значень, записаних текстом («70-80%», '
        '«до 17%»), і порожній рядок',
     'середнє 39.6 % пораховано лише на 27 записах-частках; відкинуті текстові значення в середньому більші '
     '(47 %), тож обсяг ураження занижено',
     'розібрати діапазони (середина інтервалу, «до 17%» → 17 %) і перевести все в частки: середнє '
     '42.5 %, медіана 35 %'),
    (5, 'у «spO2 без дот» є значення 0.9 — частка замість відсотків, тобто дані в різних одиницях',
     'середнє 82.05 занижене через 0.9, а висновок про «більшість пацієнтів» із середнього не '
     'випливає: про більшість говорить частка, а не середнє',
     'виправити 0.9 → 90 і звітувати медіану 84 % та частку пацієнтів із SpO₂ < 90 % — 74.4 %'),
    (6, 'коефіцієнт варіації для дат — інтервальної шкали з умовним нулем',
     'CV ділить на середнє «днів від 1970 року»; від нуля Excel (1899) ті самі дати дають 0.254 %, '
     'тож 0.595 % — артефакт вибору початку відліку',
     'для інтервальної шкали — стандартне відхилення чи розмах у днях: std 113 днів, період '
     '25.02.2021 – 03.06.2022'),
    (7, 'кореляцію Пірсона застосовано до двох номінальних ознак (стать × результат), закодованих 0/1',
     '0.177 не «близько до нуля»: χ² = 3.78, майже критичне 3.84, а жінки помирали частіше '
     '(57 % проти 39 %); висновок «зв’язку немає» з цього числа не випливає',
     'таблиця спряженості, χ² і коефіцієнт Крамера V = 0.177 — слабкий зв’язок на межі значущості'),
    (8, 'min-max нормалізацію застосовано до всіх 40 числових стовпців без перевірки шкали '
        'й варіативності',
     'серед них код «помер/жив» (номінальна) і ФЕРИТ2 з одним значенням (max = min), де ділення '
     'на нуль дало мовчазні NaN; «усі приведено до [0, 1]» — неправда',
     'нормалізувати лише стовпці шкал відношень та інтервальних, виключити коди й ідентифікатори '
     'та перевірити max ≠ min'),
    (9, 'df.dropna() без параметрів видаляє рядок, якщо пропуск є хоч в одному з 77 стовпців',
     'пропуск є в кожному рядку (11 стовпців заповнені менш ніж наполовину), тому лишилося 0 рядків — '
     '«повністю заповнений набір» порожній',
     'dropna(subset=[потрібні стовпці]) або спершу прибрати майже порожні стовпці, а решту '
     'пропусків заповнити'),
    (10, '«Дата розтину» є лише в померлих — це витік цільової змінної; NaT перетворено на int64 '
         'як −9.2·10¹⁸',
     'r = 1.00 означає лише «дата є ↔ пацієнт помер»: розтин роблять після смерті, тож для прогнозу '
     'ця ознака недоступна',
     'виключити ознаки, що з’являються після результату, і рахувати точковий бісеріальний коефіцієнт '
     'лише для ознак, відомих при поступленні'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,середнє від кодів номінальної ознаки «помер/жив» (1 = по...,"коди 1 і 2 — лише мітки; 1.50 означає тільки, що померли...","частки та мода: летальність = (df[""помер/жив""] == 1).mea..."
1,2,частоти номінальної ознаки пораховано на неочищених мітках,"«ж» і «жін», «ч» і «чол» — та сама стать, записана по-рі...","звести мітки до першої літери після strip(): ж — 75, ч — 46"
2,3,"порядкову шкалу відсортовано за абеткою, коди взято з ал...","порядок «важкий … середній» абетковий, а середнє кодів 5...","звести правопис до 5 рівнів, задати порядок вручну (легк..."
3,4,"pd.to_numeric(errors=""coerce"") мовчки викинув 16 значень...",середнє 39.6 % пораховано лише на 27 записах-частках; ві...,"розібрати діапазони (середина інтервалу, «до 17%» → 17 %..."
4,5,у «spO2 без дот» є значення 0.9 — частка замість відсотк...,"середнє 82.05 занижене через 0.9, а висновок про «більші...",виправити 0.9 → 90 і звітувати медіану 84 % та частку па...
5,6,коефіцієнт варіації для дат — інтервальної шкали з умовн...,CV ділить на середнє «днів від 1970 року»; від нуля Exce...,для інтервальної шкали — стандартне відхилення чи розмах...
6,7,кореляцію Пірсона застосовано до двох номінальних ознак ...,"0.177 не «близько до нуля»: χ² = 3.78, майже критичне 3....","таблиця спряженості, χ² і коефіцієнт Крамера V = 0.177 —..."
7,8,min-max нормалізацію застосовано до всіх 40 числових сто...,серед них код «помер/жив» (номінальна) і ФЕРИТ2 з одним ...,нормалізувати лише стовпці шкал відношень та інтервальни...
8,9,"df.dropna() без параметрів видаляє рядок, якщо пропуск є...",пропуск є в кожному рядку (11 стовпців заповнені менш ні...,dropna(subset=[потрібні стовпці]) або спершу прибрати ма...
9,10,«Дата розтину» є лише в померлих — це витік цільової змі...,r = 1.00 означає лише «дата є ↔ пацієнт помер»: розтин р...,"виключити ознаки, що з’являються після результату, і рах..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:
помер = df['помер/жив'] == 1
print('Розділ 1')
print('   неправильно: середнє кодів =', round(df['помер/жив'].mean(), 2))
print('   правильно:   летальність =', f'{помер.mean():.1%}')
print()

абетка = df['ступінь КТ'].value_counts().sort_index()
коди_абетки = {назва: i for i, назва in enumerate(абетка.index)}
НАЗВИ_КТ = {1: 'легкий', 2: 'середній', 3: 'важкий', 4: 'вкрай важкий', 5: 'критичний'}
print('Розділ 3')
print('   неправильно: середнє абеткових кодів =', round(df['ступінь КТ'].map(коди_абетки).mean(), 2))
print('   правильно:   медіана впорядкованих рівнів =', НАЗВИ_КТ[int(кт_рівень.median())])
print()

стать = df['Стать'].astype(str).str.strip().str[0]
таблиця = pd.crosstab(стать, помер)
O = таблиця.to_numpy(dtype=float)
E = O.sum(axis=1, keepdims=True) * O.sum(axis=0, keepdims=True) / O.sum()
хі2 = ((O - E) ** 2 / E).sum()
крамер = np.sqrt(хі2 / (O.sum() * (min(O.shape) - 1)))
print('Розділ 7')
print('   неправильно: Пірсон =', round(стать.map({'ч': 0, 'ж': 1}).corr(помер.astype(int)), 3),
      '→ «зв’язку немає»')
print('   правильно:   χ² =', round(хі2, 2), '(критичне 3.84 при α = 0.05) | V Крамера =', round(крамер, 3))
print(таблиця)

Розділ 1
   неправильно: середнє кодів = 1.5
   правильно:   летальність = 50.4%

Розділ 3
   неправильно: середнє абеткових кодів = 5.9
   правильно:   медіана впорядкованих рівнів = середній

Розділ 7
   неправильно: Пірсон = 0.177 → «зв’язку немає»
   правильно:   χ² = 3.78 (критичне 3.84 при α = 0.05) | V Крамера = 0.177
помер/жив  False  True 
Стать                  
ж             32     43
ч             28     18


---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ускл_є = df['Ускладнення'].notna()
ВИСНОВКИ = [
    f'«Ускладнення» заповнено в {ускл_є.sum()} записі з {len(df)}, і всі ці пацієнти померли '
    f'({(ускл_є & помер).sum()} з {ускл_є.sum()}); у живих поле порожнє, тож це запис про причину смерті, а не ознака при '
    'поступленні, і в моделі летальності він дав би витік цільової змінної.',
    f'«МНВ2» без очищення дає середнє {мнв.mean():.2f} через неможливе значення {мнв.max():.0f} і дату Excel '
    f'замість числа, тоді як медіана {мнв.median():.2f} — для цього стовпця треба звітувати медіану.',
    f'«Д-дим1»: {(д_дим == 10000).sum()} з {д_дим.notna().sum()} числових значень дорівнюють рівно 10000 — '
    f'це верхня межа аналізатора, тож середнє {д_дим.mean():.0f} занижене, і доречніше брати медіану '
    f'{д_дим.median():.0f}.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. «Ускладнення» заповнено в 61 записі з 121, і всі ці пацієнти померли (61 з 61); у живих поле порожнє, тож це запис про причину смерті, а не ознака при поступленні, і в моделі летальності він дав би витік цільової змінної.
2. «МНВ2» без очищення дає середнє 4.41 через неможливе значення 239 і дату Excel замість числа, тоді як медіана 1.20 — для цього стовпця треба звітувати медіану.
3. «Д-дим1»: 11 з 57 числових значень дорівнюють рівно 10000 — це верхня межа аналізатора, тож середнє 3567 занижене, і доречніше брати медіану 1581.


In [21]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
поч, кін = pd.Timestamp('2021-09-01'), pd.Timestamp('2022-04-02')
від_1970 = (кін - pd.Timestamp('1970-01-01')).days / (поч - pd.Timestamp('1970-01-01')).days
від_1900 = (кін - pd.Timestamp('1899-12-30')).days / (поч - pd.Timestamp('1899-12-30')).days
ліжко = pd.to_numeric(df['Ліжко днів'], errors='coerce')
ВІДПОВІДЬ = (
    'В інтервальній шкалі нуль умовний, тому мають сенс лише різниці, а у шкалі відношень нуль '
    'природний, тому мають сенс і відношення. Приклад інтервальної шкали — дата «Поступлення»: між '
    f'01.09.2021 і 02.04.2022 минуло {(кін - поч).days} днів за будь-якого календаря, а «відношення дат» '
    f'залежить від початку відліку: {від_1970:.4f} від 1970 року і {від_1900:.4f} від нуля Excel, '
    'тож воно нічого не означає. Приклад шкали відношень — «Ліжко днів»: 0 днів означає відсутність '
    'перебування, а 20 днів справді вдвічі довше за 10. Операція, допустима для шкали відношень і '
    'недопустима для інтервальної, — ділення: відношення, відсотки, коефіцієнт варіації '
    f'({ліжко.std() / ліжко.mean():.2f} для ліжко-днів) і геометричне середнє.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
У чому різниця між шкалою інтервалів і шкалою відношень? Знайдіть у файлі по одному прикладу кожної та поясніть, яка операція допустима для другої, але не для першої.

В інтервальній шкалі нуль умовний, тому мають сенс лише різниці, а у шкалі відношень нуль природний, тому мають сенс і відношення. Приклад інтервальної шкали — дата «Поступлення»: між 01.09.2021 і 02.04.2022 минуло 213 днів за будь-якого календаря, а «відношення дат» залежить від початку відліку: 1.0113 від 1970 року і 1.0048 від нуля Excel, тож воно нічого не означає. Приклад шкали відношень — «Ліжко днів»: 0 днів означає відсутність перебування, а 20 днів справді вдвічі довше за 10. Операція, допустима для шкали відношень і недопустима для інтервальної, — ділення: відношення, відсотки, коефіцієнт варіації (0.72 для ліжко-днів) і геометричне середнє.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [29]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'текст, перероблено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, перероблено',
    'етап 7 — висновки й відповідь':   'текст, перероблено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'перевірено помилки в коді помилкового аналізу, проаналізовано ексель таблицю з даними, виправлено помилки які були допущені протягом виконання'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [30]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 4 готова до захисту.
Левченко Денис, ШІ-33


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.